# channel_no_conc Batch Workflow

批量入口 notebook：扫描一个目录下第一层的主配置 JSON，展开其中引用的全部模板，并逐个运行 `channel_no_conc` compare。


In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start=None):
    cwd = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "examples" / "neuron_compare" / "channel_no_conc" / "workflows" / "workflow_api.py").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


REPO_ROOT = find_repo_root()
CHANNEL_NO_CONC_ROOT = REPO_ROOT / "examples" / "neuron_compare" / "channel_no_conc"
WORKFLOWS_ROOT = CHANNEL_NO_CONC_ROOT / "workflows"
if str(WORKFLOWS_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKFLOWS_ROOT))

import workflow_api
import brainstate
brainstate.environ.set(precision=64)


## Parameters

| 参数 | 含义 | 常见取值 |
|---|---|---|
| `config_dir` | 批量扫描目录；只扫描第一层 `*.json`，每个 config 会跑完自己声明的全部模板。 | `CHANNEL_NO_CONC_ROOT / "configs"` |
| `summary_dir` | 本次 batch run 的输出根目录。 | `None` 自动生成为 `results/batch_runs/YYMMDD_HHMMSS/`；或手动指定路径 |
| `plot_cases` | 是否在每个 config 的模板子目录里生成图产物。 | `True` / `False` |


In [3]:
config_dir = CHANNEL_NO_CONC_ROOT / "configs" / "ma24_pc"  # 批量扫描目录；每个 config 会跑完自己声明的全部模板
summary_dir = None  # 本次 batch run 的输出根目录；None 时自动使用 YYMMDD_HHMMSS 命名
plot_cases = True # 是否在每个 config 的模板子目录里生成图产物


## Discover Configs And Prepare Batch Directory


In [4]:
config_records = workflow_api.discover_batch_configs(config_dir)
batch_run_id = workflow_api.make_batch_run_id()
resolved_summary_dir = workflow_api.default_batch_run_output_dir(
    batch_run_id=batch_run_id,
    summary_dir=summary_dir,
)
configs_out_dir = resolved_summary_dir / "configs"
configs_out_dir.mkdir(parents=True, exist_ok=True)

preview_df = pd.DataFrame(
    [
        {
            "config_name": row["config_name"],
            "config_path": str(row["config_path"]),
            "n_templates": row["n_templates"],
            "template_names": ", ".join(row["template_names"]),
            "config_out_dir": str(configs_out_dir / row["config_name"]),
        }
        for row in config_records
    ]
)
print("batch_run_id:", batch_run_id)
print("resolved_summary_dir:", resolved_summary_dir)
display(preview_df)


batch_run_id: 260504_000359
resolved_summary_dir: /home/merveille/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260504_000359


,config_name,config_path,n_templates,template_names,config_out_dir
0,hcn1_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,3,"vinit_celsius, dc, ac",/home/merveille/braincell/examples/neuron_comp...
1,kir2p3_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,3,"vinit_celsius, dc, ac",/home/merveille/braincell/examples/neuron_comp...
2,kv1p1_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,3,"vinit_celsius, dc, ac",/home/merveille/braincell/examples/neuron_comp...
3,kv3p4_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,3,"vinit_celsius, dc, ac",/home/merveille/braincell/examples/neuron_comp...
4,kv4p3_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,3,"vinit_celsius, dc, ac",/home/merveille/braincell/examples/neuron_comp...


## Run Each Config


In [6]:
config_run_infos = []
for record in config_records:
    config_out_dir = configs_out_dir / str(record["config_name"])
    run_info = workflow_api.run_notebook_config_workflow(
        config_path=record["config_path"],
        out_dir=config_out_dir,
        plot=plot_cases,
        expand_only=False,
        raise_on_failure=False,
    )
    config_run_infos.append(run_info)

batch_summary = workflow_api.write_batch_summary_artifacts(
    config_dir=config_dir,
    summary_dir=resolved_summary_dir,
    batch_run_id=batch_run_id,
    config_records=config_records,
    config_run_infos=config_run_infos,
    plot_cases=plot_cases,
)
print("batch_manifest:", batch_summary["manifest_path"])
print("batch_observable_summary_csv:", batch_summary["batch_observable_summary_path"])
print("batch_observable_summary_json:", batch_summary["batch_observable_summary_json_path"])
print("batch_failures_csv:", batch_summary["batch_failures_path"])


batch_manifest: /home/merveille/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260504_000359/batch_manifest.json
batch_observable_summary_csv: /home/merveille/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260504_000359/batch_observable_summary.csv
batch_observable_summary_json: /home/merveille/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260504_000359/batch_observable_summary.json
batch_failures_csv: /home/merveille/braincell/examples/neuron_compare/channel_no_conc/results/batch_runs/260504_000359/batch_failures.csv


## Load Batch Summary Tables


In [7]:
runs_df = pd.read_csv(batch_summary["config_runs_path"]).sort_values(["batch_status", "config_name"]).reset_index(drop=True)
observables_df = pd.read_csv(batch_summary["batch_observable_summary_path"]).sort_values(["observable", "config_name"]).reset_index(drop=True)
failures_df = pd.read_csv(batch_summary["batch_failures_path"]).reset_index(drop=True)

display(observables_df)
display(failures_df)


,config_name,config_path,batch_status,observable,n_cases,mae_mean,rmse_mean,max_abs_max,rel_mae_pct_mean
0,hcn1_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,current.ix,16,7.053644e-17,8.299945e-17,4.424629e-16,9.271136e-11
1,kir2p3_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,current.ix,16,4.709992e-07,5.704895e-07,7.511425e-06,2.443626e-02
2,kv1p1_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,current.ix,16,8.255489e-16,1.106758e-15,1.189153e-14,2.754716e-10
3,kv3p4_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,current.ix,16,8.973919e-16,1.333263e-15,3.274638e-14,1.221972e-10
4,kv4p3_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,current.ix,16,2.847215e-07,3.689298e-07,7.820842e-06,4.783145e-02
5,kv4p3_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,gates.a,16,9.537955e-05,1.815165e-04,6.430725e-03,8.854804e-02
6,kv4p3_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,gates.b,16,1.948846e-05,5.583734e-05,3.933530e-03,1.144000e-02
7,kir2p3_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,gates.d,16,4.346547e-05,5.220582e-05,2.324367e-04,2.598651e-02
8,hcn1_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,gates.h,16,2.076633e-15,2.845386e-15,1.472780e-14,1.228486e-11
9,kv3p4_ma24_pc,/home/merveille/braincell/examples/neuron_comp...,ok,gates.h,16,4.914812e-15,6.815490e-15,1.175726e-13,4.965157e-13


,config_name,config_path,batch_status,template_name,template_path,out_dir,n_failed_cases,error_message


如果某个 config 需要继续做 template / case 级钻取，进入该 batch run 目录下的 `configs/<config_name>/`，再用 [workflow.ipynb](/home/swl/braincell/examples/neuron_compare/channel_no_conc/workflows/workflow.ipynb) 对单个 config 继续分析。
